In [12]:
!pip install ultralytics gradio -q

from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import display, Image
import gradio as gr
print("✅ Setup complete!")

✅ Setup complete!


In [2]:

model = YOLO("yolo11s-pose.pt")

print("✅ Pose model loaded!")

✅ Pose model loaded!


In [8]:
def is_falling(keypoints):
    """Advanced fall detection using pose keypoints"""
    if len(keypoints) < 17:
        return False

    # Key points: 0=nose, 11=left shoulder, 12=right shoulder, 15=left ankle, 16=right ankle
    head_y = keypoints[0][1].item()
    shoulder_y = (keypoints[11][1].item() + keypoints[12][1].item()) / 2
    ankle_y = (keypoints[15][1].item() + keypoints[16][1].item()) / 2

    # If head and ankles are close in height → person is lying down
    height_diff = abs(head_y - ankle_y)
    shoulder_height = abs(shoulder_y - ankle_y)

    if height_diff < 90 and shoulder_height < 120:   # Tuned thresholds
        return True
    return False

In [9]:
def detect_fall(image):
    results = model.predict(image, conf=0.5, verbose=False)
    annotated = results[0].plot()

    fall_detected = False
    for result in results:
        if result.keypoints is not None:
            kpts = result.keypoints.data[0]
            if is_falling(kpts):
                fall_detected = True
                cv2.putText(annotated, "🚨 FALL DETECTED!", (50, 100),
                          cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 0, 255), 6)
                cv2.putText(annotated, "Call for help!", (50, 160),
                          cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    status = "🚨 Fall Detected! Call for help" if fall_detected else "✅ No Fall Detected"
    return annotated, status

In [10]:
# Test image
test_url = "https://ultralytics.com/images/bus.jpg"

# Run detection
annotated_img, status = detect_fall(test_url)

# Show image properly
display(Image(annotated_img))
print("Status:", status)

Output hidden; open in https://colab.research.google.com to view.

In [14]:
interface = gr.Interface(
    fn=detect_fall,
    inputs=gr.Image(type="pil", label="Upload Image (or use webcam)"),
    outputs=[
        gr.Image(type="pil", label="Detection Result"),
        gr.Textbox(label="Alert Status")
    ],
    title="🛡️ Advanced Fall Detection System",
    description="Real-time fall detection using YOLO11 Pose Estimation. Ideal for elderly care and smart homes.",
    examples=[
        ["examples/fall_example.jpg"],
        ["examples/normal_example.jpg"]
    ],
    live=True
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b9e88fbb626b6de8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
